In [1]:
from typing import List, Optional, Union
from pydantic import BaseModel, Field, HttpUrl, validator
from datetime import datetime

# --- Deployment Sub-Models ---
class CloudParams(BaseModel):
    type: str  # enum: azure_iotc, aws, custom
    app_url: Optional[str] = None

class Deployment(BaseModel):
    uuid: str
    display_name: str
    description: str
    last_update_time: Optional[str] = None
    last_deploy_result: Optional[str] = None
    cloud_params: CloudParams
    # Note: In your project file, leaf/gateway are direct children, 
    # though your schema expected a 'configuration' wrapper
    leaf: List[dict] = []
    gateway: List[dict] = []

# --- Model Sub-Models ---
class ModelMetadata(BaseModel):
    type: str  # enum: classifier, anomaly_detector, predictor
    classes: List[str]

class AIModel(BaseModel):
    uuid: str
    name: str
    description: Optional[str] = None
    creation_time: Optional[str] = None # Using str because your project file has nulls
    metadata: ModelMetadata
    dataset: dict
    target: dict
    training: Optional[dict] = None # Marked optional as project file uses 'data_sufficiency' instead

# --- Main Project Model ---
class AIProject(BaseModel):
    ai_project_name: str
    display_name: str
    description: str
    uuid: str
    version: str = Field(..., pattern=r"^[0-9].[0-9].[0-9]$")
    creation_time: Optional[str] = None
    last_update_time: Optional[str] = None
    models: List[AIModel]
    deployments: List[Deployment]

In [3]:
import json
from pydantic import ValidationError

# Load the raw JSON data
with open('ai_get_started_smart_asset_tracking_md_mlc.json', 'r') as f:
    raw_data = json.load(f)

try:
    # This single line validates AND converts the JSON into a Python Object
    project = AIProject(**raw_data)
    
    print(f"Success! Loaded Project: {project.display_name}")
    print(f"Number of Models: {len(project.models)}")
    
except ValidationError as e:
    print("Validation failed!")
    print(e.json())

Success! Loaded Project: Smart Asset Tracking MD
Number of Models: 1
